# Practical P10: Implementing Streaming Responses
**Learning Outcome**: Implement streaming for any LLM API provider to improve user experience.

## Part 1: Buffered vs Streaming Responses Flowchart
Streaming sends tokens incrementally as they are generated by the model using Server-Sent Events (SSE). This reduces time-to-first-token (TTFT) and makes applications feel faster.

```mermaid
graph TD
  subgraph Buffered Response
    A[Request] --> B[Model Computes...] --> C[Wait for complete response] --> D[Display raw text]  end
  subgraph Streaming Response
    E[Request] --> F[Model Computes...] --> G[Token 1] --> H[Print Token 1] --> I[Token 2] --> J[Print Token 2] --> K[Stop Stream]  end
```


## Part 2: Implementing Python Generators for Streaming
Let's write a generator function that yields tokens sequentially with a short latency delay, simulating real streaming API loops.


In [1]:
import time
import sys

def mock_stream_generator(prompt):
    tokens = f"Streaming chunks responding to your prompt '{prompt}'. Completed successfully.".split()
    for t in tokens:
        yield t + ' '
        time.sleep(0.1) # Simulate generation latency

print('Starting stream: ')
for chunk in mock_stream_generator('Explain streaming'):
    # Print token immediately without buffer or newline
    sys.stdout.write(chunk)
    sys.stdout.flush()
print('\nStream Finished.')


Starting stream: 
Streaming chunks responding to your prompt 'Explain streaming'. Completed successfully. 
Stream Finished.


## Hands-On Exercise
**Task**: Write a benchmark function `time_stream_vs_buffered(prompt)` that measures the time difference between receiving the *first token* (TTFT) and receiving the *full response* (Total Time).
Verify the results on different simulated prompt lengths.


In [2]:
# TODO: Complete time_stream_vs_buffered
def time_stream_vs_buffered(prompt):
    # Simulate buffered load time
    t_start = time.time()
    time.sleep(1.0) # Model execution buffer
    total_time_buffered = time.time() - t_start
    
    # Simulate streaming
    t_start_stream = time.time()
    stream = mock_stream_generator(prompt)
    first_token = next(stream)
    ttft = time.time() - t_start_stream
    
    # Consume rest of stream
    for chunk in stream: pass
    total_time_stream = time.time() - t_start_stream
    
    print(f'Buffered Total Time: {total_time_buffered:.2f}s')
    print(f'Streaming TTFT:      {ttft:.2f}s  (Time to first token)')
    print(f'Streaming Total:     {total_time_stream:.2f}s')
    print(f'Perceived speedup:   {total_time_buffered / ttft:.1f}x faster start!')

time_stream_vs_buffered('Benchmarking metrics')


Buffered Total Time: 1.00s
Streaming TTFT:      0.00s  (Time to first token)
Streaming Total:     1.04s
Perceived speedup:   49967.7x faster start!
